In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ==========================================
# 1. Загрузка и подготовка данных
# ==========================================

df = pd.read_csv("dataset_var8_num2.csv", header=None, sep=';')

# ✅ ВАЖНО: переименуем колонки в строки, чтобы не было конфликтов с индексацией
df.columns = [f"c{i}" for i in range(df.shape[1])]

print("======= Информация о датасете =======")
print(df.info())

print("\n======= Пропуски (NaN) =======")
print(df.isna().sum())

target_col = df.columns[-1]   # последний столбец — класс
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

print("\n======= Классы (распределение) =======")
print(y.value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title("Распределение классов (target)")
plt.grid(True, axis='y', alpha=0.3)
plt.show()


# ==========================================
# 2. Анализ данных + корреляции + выбросы
# ==========================================

# Заполним NaN для аналитики (медианой), чтобы можно было посчитать корреляции
X_eda = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X),
    columns=X.columns
)
eda = X_eda.copy()
eda["target"] = y

print("\n======= Статистика признаков =======")
print(eda.describe())

# Матрица корреляций (включая target)
plt.figure(figsize=(10, 8))
sns.heatmap(eda.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Матрица корреляций")
plt.show()

corr_target = eda.corr()["target"].sort_values(ascending=False)
print("\n======= Корреляция с target =======")
print(corr_target)

# Выбросы (IQR) — просто оценка количества
print("\n======= Оценка выбросов (IQR) =======")
out_info = []
for col in X_eda.columns:
    s = X_eda[col]
    q1, q3 = np.percentile(s, 25), np.percentile(s, 75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out_cnt = ((s < low) | (s > high)).sum()
    out_info.append((col, out_cnt))
out_df = pd.DataFrame(out_info, columns=["feature", "outliers_count"]).sort_values("outliers_count", ascending=False)
print(out_df)

# ------------------------------------------
# 2.1 Удаление дубликатов признаков (если есть одинаковые столбцы)
# ------------------------------------------
dup_cols_mask = X_eda.T.duplicated()
dup_cols = X_eda.columns[dup_cols_mask].tolist()

print("\n======= Проверка на дубликаты столбцов =======")
print("Дубликаты:", dup_cols)

if dup_cols:
    X = X.drop(columns=dup_cols)
    X_eda = X_eda.drop(columns=dup_cols)

# ------------------------------------------
# 2.2 Удаление слабых признаков по корреляции с target (можно менять порог)
# ------------------------------------------
corr_threshold = 0.05
weak = [c for c in X_eda.columns if abs(corr_target.get(c, 0.0)) < corr_threshold]

print("\n======= Удаление слабых признаков =======")
print("Порог |corr| < ", corr_threshold)
print("Удаляем:", weak)

# Удаляем только если не убили всё
if len(weak) < X.shape[1]:
    X = X.drop(columns=weak)


# ==========================================
# 3. Разбиение 60/40
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

print("\nTrain:", X_train.shape, "Test:", X_test.shape)


# ==========================================
# 4. Модели + 5. Подбор параметров
# ==========================================

# ---- RandomForest (скейлинг не нужен)
rf_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(random_state=42))
])

rf_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced"]
}

rf_search = GridSearchCV(
    rf_pipe,
    rf_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

print("\n======= RandomForest best params =======")
print(rf_search.best_params_)
print("Best CV f1_macro:", rf_search.best_score_)


# ---- KNN (нужен скейлинг)
knn_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

knn_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]  # 1=manhattan, 2=euclidean
}

knn_search = GridSearchCV(
    knn_pipe,
    knn_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)
knn_search.fit(X_train, y_train)
best_knn = knn_search.best_estimator_

print("\n======= KNN best params =======")
print(knn_search.best_params_)
print("Best CV f1_macro:", knn_search.best_score_)


# ==========================================
# 6. Метрики и сравнение моделей
# ==========================================

def eval_model(name, model):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1m  = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"\n======= {name} =======")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision(macro): {prec:.4f}")
    print(f"Recall(macro): {rec:.4f}")
    print(f"F1(macro): {f1m:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion matrix: {name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    return f1m

f1_rf = eval_model("RandomForestClassifier", best_rf)
f1_knn = eval_model("KNeighborsClassifier", best_knn)

print("\n======= Итог =======")
if f1_rf >= f1_knn:
    print(f"Лучшая модель: RandomForestClassifier (F1_macro={f1_rf:.4f})")
else:
    print(f"Лучшая модель: KNeighborsClassifier (F1_macro={f1_knn:.4f})")

# Важность признаков (только для RF)
rf_model = best_rf.named_steps["model"]
importances = rf_model.feature_importances_
print("\n======= Важность признаков (RandomForest) =======")
for col, imp in sorted(zip(X.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col}: {imp:.4f}")
